# Task 2: Player segment classification (4 classes, macro F1)

**Goal.** Predict `segment` (0 Casual, 1 Grinder, 2 Social, 3 Whale). Metric: macro F1.

**Final method (as submitted to Kaggle).** On the corrected Task 2 data (released 2026-11-12):
1. Drop the `random_metric*` columns (pure noise); categorical columns as pandas categories.
2. LightGBM multiclass, 5-fold stratified CV (seed 342); test probability = mean of the fold models.
3. Class weights multiplied onto the probabilities, tuned by coordinate search on the OOF predictions to maximise macro F1.

**Result.** OOF macro F1 about 0.751 (argmax) and 0.752 (weighted). Logistic regression reaches 0.7506, so the ceiling is the data. Output: `../outputs/task2_pred.npy`.

In [1]:
import os, re, sys, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

# ---- configuration -------------------------------------------------------------------------------
# DATA = folder of the Kaggle download that holds 01_binary_classification/ ... 05_anomaly_detection/ and sample_submission.csv
DATA = Path(os.environ.get("CPE342_DATA", "")).expanduser()
assert (DATA / "01_binary_classification").exists(), "Set the environment variable CPE342_DATA to the Kaggle download folder (see ../README.md)"
OUT = Path(os.environ.get("CPE342_OUT", "../outputs")); OUT.mkdir(parents=True, exist_ok=True)   # shared by all notebooks
SEED = 342                                   # every random split / model seed in the project
NJOBS = int(os.environ.get("NJOBS", 16))     # LightGBM threads
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

## 1. Data and quick EDA

In [2]:
d2 = DATA / "02_multiclass_classification"
tr, te = pd.read_csv(d2 / "train.csv"), pd.read_csv(d2 / "test.csv")
y = tr.segment.astype(int).values
cats = [c for c in tr.columns if c not in ("player_id", "segment") and not pd.api.types.is_numeric_dtype(tr[c])]
feat = [c for c in tr.columns if c not in ("player_id", "segment") and not c.startswith("random_metric")]
print(f"train {len(tr):,} rows, test {len(te):,} rows | features used {len(feat)} (dropped {sum(c.startswith('random_metric') for c in tr.columns)} noise columns) | categorical: {cats}")
print("class shares:", np.round(np.bincount(y) / len(y), 3).tolist())
print("columns with missing values (share):", tr[feat].isna().mean().loc[lambda s: s > 0].round(3).to_dict())

train 104,000 rows, test 25,889 rows | features used 41 (dropped 3 noise columns) | categorical: ['region', 'platform', 'device_type', 'payment_method', 'language', 'account_status', 'player_type_tag', 'engagement_level', 'loyalty_tier', 'skill_tier']
class shares: [0.394, 0.25, 0.202, 0.154]
columns with missing values (share): {'play_frequency': 0.077, 'avg_session_duration': 0.077, 'total_playtime_hours': 0.077, 'login_streak': 0.076, 'days_since_last_login': 0.077, 'total_spending_thb': 0.077, 'avg_monthly_spending': 0.078, 'spending_frequency': 0.077, 'friend_count': 0.077, 'team_play_percentage': 0.078, 'chat_activity_score': 0.077, 'friend_invites_sent': 0.077, 'gifts_sent_received': 0.076, 'ranked_participation_rate': 0.078, 'tournament_entries': 0.077, 'competitive_rank': 0.077, 'win_rate_ranked': 0.076, 'watches_esports': 0.077, 'achievement_completion_rate': 0.075, 'collection_progress': 0.077, 'rare_items_count': 0.077, 'speed_of_progression': 0.077, 'item_type_preference_c

## 2. Cross-validated LightGBM

In [3]:
def prep(d):
    X = d[feat].copy()
    for c in cats:
        if c in X: X[c] = pd.Categorical(X[c], categories=sorted(tr[c].dropna().unique()))
    return X
X, Xte = prep(tr), prep(te)
P = dict(objective="multiclass", num_class=4, learning_rate=0.03, num_leaves=31, min_child_samples=40, feature_fraction=0.7,
         bagging_fraction=0.8, bagging_freq=1, lambda_l2=5.0, verbose=-1, seed=SEED, n_jobs=NJOBS)
oof, pte = np.zeros((len(X), 4)), np.zeros((len(Xte), 4))
for k, (a, b) in enumerate(StratifiedKFold(5, shuffle=True, random_state=SEED).split(X, y)):
    m = lgb.train(P, lgb.Dataset(X.iloc[a], y[a]), 3000, valid_sets=[lgb.Dataset(X.iloc[b], y[b])], callbacks=[lgb.early_stopping(150, verbose=False)])
    oof[b] = m.predict(X.iloc[b], num_iteration=m.best_iteration); pte += m.predict(Xte, num_iteration=m.best_iteration) / 5
    print(f"  fold {k}: trees {m.best_iteration}, macro F1 {f1_score(y[b], oof[b].argmax(1), average='macro'):.4f}", flush=True)

  fold 0: trees 610, macro F1 0.7523


  fold 1: trees 471, macro F1 0.7490


  fold 2: trees 490, macro F1 0.7521


  fold 3: trees 527, macro F1 0.7513


  fold 4: trees 496, macro F1 0.7514


## 3. Class weights for macro F1 (coordinate search on the OOF predictions)

In [4]:
w = np.ones(4); grid = np.linspace(0.6, 1.8, 25)
for _ in range(3):
    for c in range(4):
        best = (f1_score(y, (oof * w).argmax(1), average="macro"), w[c])
        for g in grid:
            w2 = w.copy(); w2[c] = g; s = f1_score(y, (oof * w2).argmax(1), average="macro")
            if s > best[0]: best = (s, g)
        w[c] = best[1]
print(f"OOF macro F1: argmax {f1_score(y, oof.argmax(1), average='macro'):.4f} | weighted {f1_score(y, (oof * w).argmax(1), average='macro'):.4f} | weights {np.round(w, 2).tolist()}")
pred = (pte * w).argmax(1)
np.save(OUT / "task2_test_prob.npy", pte); np.save(OUT / "task2_pred.npy", pred)
print("test class shares:", np.round(np.bincount(pred, minlength=4) / len(pred), 3).tolist(), "| train shares:", np.round(np.bincount(y) / len(y), 3).tolist())

OOF macro F1: argmax 0.7512 | weighted 0.7524 | weights [0.85, 0.9, 1.0, 1.0]
test class shares: [0.416, 0.249, 0.195, 0.14] | train shares: [0.394, 0.25, 0.202, 0.154]
